# Homework - Annotated Notebook

This notebook includes step-by-step markdown sections and English code comments.

Activate virtual env: source .venv/bin/activate

## Step 1 - Import required libraries

In [1]:
# Import required libraries.
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

## Step 2 - Create a Spark session

In [8]:
# Create a Spark session.
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

## Step 3 - Run the next processing step

In [9]:
# Run the next processing step.
spark.version

'4.1.1'

## Step 4 - Download and inspect local files and raw data

In [26]:
!mkdir -p ./data && if [ ! -f ./data/yellow_tripdata_2025-11.parquet ]; then wget -P ./data https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet; fi

In [27]:
# Inspect local files and raw data.
!ls -lh ./data/yellow_tripdata_2025-11.parquet

-rw-r--r--@ 1 rgctechfi  staff    68M Dec 19 16:51 ./data/yellow_tripdata_2025-11.parquet


## Step 5 - Load data into a Spark DataFrame

In [32]:
# Load data into a Spark DataFrame.
df = spark.read \
    .option("header", "true") \
    .csv('./data/yellow_tripdata_2025-11.parquet')

df = df.repartition(4)

## Step 7 - Load data into a Spark DataFrame

In [ ]:
# Load data into a Spark DataFrame.
df = spark.read.parquet('./data/yellow_tripdata_2025-11.parquet')
df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

# Partitioned

In [39]:
df.repartition(4).write.parquet('partitioned/yellow_2025_11_repartitioned')


AnalysisException: [PATH_ALREADY_EXISTS] Path file:/Users/rgctechfi/Projects/spark/project/partitioned/yellow_2025_11_repartitioned already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

- _SUCCESS file (which indicates that the job finished successfully).

- 4 files starting with part-0000x... and ending with .parquet.

Size : 25 MiB for each file

In [ ]:
ls -lah ./data/yellow_2025_11_repartitioned #h for displaying size but not in byte

total 207368
drwxr-xr-x@ 12 rgctechfi  staff   384B Mar  5 16:23 ./
drwxr-xr-x@  4 rgctechfi  staff   128B Mar  5 16:30 ../
-rw-r--r--@  1 rgctechfi  staff     8B Mar  5 16:23 ._SUCCESS.crc
-rw-r--r--@  1 rgctechfi  staff   195K Mar  5 16:23 .part-00000-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet.crc
-rw-r--r--@  1 rgctechfi  staff   195K Mar  5 16:23 .part-00001-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet.crc
-rw-r--r--@  1 rgctechfi  staff   195K Mar  5 16:23 .part-00002-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet.crc
-rw-r--r--@  1 rgctechfi  staff   195K Mar  5 16:23 .part-00003-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet.crc
-rw-r--r--@  1 rgctechfi  staff     0B Mar  5 16:23 _SUCCESS
-rw-r--r--@  1 rgctechfi  staff    24M Mar  5 16:23 part-00000-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet
-rw-r--r--@  1 rgctechfi  staff    24M Mar  5 16:23 part-00001-be1ed0fe-87fa-4a85-8fa5-9bb11f34a9f4-c000.snappy.parquet
-rw-r--r-

## Step 8 - Import required libraries

In [37]:
# Import required libraries.
from pyspark.sql import functions as F

## Step 9 - Run the next processing step

In [52]:
# Run the next processing step.
df \
    .withColumn('tpep_pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("tpep_pickup_date = '2021-11-15'") \
    .count()

0

## Step 10 - Register a temporary SQL view

In [53]:
# Register a temporary SQL view.
df.registerTempTable('yellow_2025_11')

/Users/rgctechfi/Projects/spark/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


## Step 11 - Run a Spark SQL query - counting records

**Q3**: How many taxi trips were there on February 15?

In [56]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



Q3: 162,604

**Q4**: Longest trip for each day

## Step 12 - Run the next processing step

In [29]:
# Run the next processing step.
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

## Step 13 - Run the next processing step

In [36]:
# Run the next processing step.
df \
    .withColumn('duration', df.dropoff_datetime.cast('long') - df.pickup_datetime.cast('long')) \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

+-----------+-------------+
|pickup_date|max(duration)|
+-----------+-------------+
| 2021-02-11|        75540|
| 2021-02-17|        57221|
| 2021-02-20|        44039|
| 2021-02-03|        40653|
| 2021-02-19|        37577|
+-----------+-------------+




[Stage 38:==================================================>   (187 + 4) / 200]



## Step 14 - Run a Spark SQL query

In [38]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    to_date(pickup_datetime) AS pickup_date,
    MAX((CAST(dropoff_datetime AS LONG) - CAST(pickup_datetime AS LONG)) / 60) AS duration
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

+-----------+-----------------+
|pickup_date|         duration|
+-----------+-----------------+
| 2021-02-11|           1259.0|
| 2021-02-17|953.6833333333333|
| 2021-02-20|733.9833333333333|
| 2021-02-03|           677.55|
| 2021-02-19|626.2833333333333|
| 2021-02-25|            583.5|
| 2021-02-18|576.8666666666667|
| 2021-02-10|569.4833333333333|
| 2021-02-21|           537.05|
| 2021-02-09|534.7833333333333|
+-----------+-----------------+




[Stage 44:================================================>     (180 + 4) / 200]



**Q5**: Most frequent `dispatching_base_num`

How many stages this spark job has?



## Step 15 - Run a Spark SQL query

In [44]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    dispatching_base_num,
    COUNT(1)
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()


[Stage 73:>                                                         (0 + 4) / 4]



+--------------------+--------+
|dispatching_base_num|count(1)|
+--------------------+--------+
|              B02510| 3233664|
|              B02764|  965568|
|              B02872|  882689|
|              B02875|  685390|
|              B02765|  559768|
+--------------------+--------+




[Stage 74:===================================================>  (189 + 5) / 200]



## Step 16 - Run the next processing step

In [46]:
# Run the next processing step.
df \
    .groupBy('dispatching_base_num') \
        .count() \
    .orderBy('count', ascending=False) \
    .limit(5) \
    .show()


[Stage 86:>                                                         (0 + 4) / 4]



+--------------------+-------+
|dispatching_base_num|  count|
+--------------------+-------+
|              B02510|3233664|
|              B02764| 965568|
|              B02872| 882689|
|              B02875| 685390|
|              B02765| 559768|
+--------------------+-------+




[Stage 87:===========================================>          (161 + 5) / 200]



**Q6**: Most common locations pair

## Step 17 - Load data into a Spark DataFrame

In [47]:
# Load data into a Spark DataFrame.
df_zones = spark.read.parquet('zones')

## Step 18 - Run the next processing step

In [49]:
# Run the next processing step.
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

## Step 19 - Run the next processing step

In [51]:
# Run the next processing step.
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

## Step 20 - Register a temporary SQL view

In [50]:
# Register a temporary SQL view.
df_zones.registerTempTable('zones')

## Step 21 - Run a Spark SQL query

In [57]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    fhvhv_2021_02 fhv LEFT JOIN zones pul ON fhv.PULocationID = pul.LocationID
                      LEFT JOIN zones dol ON fhv.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|East New York / E...|   45041|
|Borough Park / Bo...|   37329|
| Canarsie / Canarsie|   28026|
|Crown Heights Nor...|   25976|
|Bay Ridge / Bay R...|   17934|
+--------------------+--------+



## Step 22 - Run this processing step

In [ ]:
# Run this processing step.
